# NCBI Virus Dataset Creation

Author: Alexander Maksiaev

Purpose: Create dataset using NCBI and relabeling sequences. 

Notes:
* This file must be in the same folder as "utils.py"

## Housekeeping

### Libraries

In [1]:
# Libraries

import os
import pandas as pd
import dateutil
import re
import shutil 
import numpy as np
import importlib
from pathlib import Path
import utils  
importlib.reload(utils)
from utils import * # If changing utils, must restart this file for changes to take effect

### Input

Change these variables to what you need. The locations must be a string with each continent separated by a comma and space. The serotypes and genotypes must be lists. The dates must be strings. 

In [2]:
# Dates and locations
locations = "Antarctica, North America, South America"
serotypes = ["H5N1"]
# genotypes = ["A3"]
start_date = "11-01-2021"
end_date = "08-07-2026"

# Make a date range
date_range = start_date + "--" + end_date


### Paths

Make sure these paths fit your schema. Here is the general tree structure: </br>

* Avian Flu: This is where the code in this repository is housed. "references" is a subdirectory here.
* Avian Flu Files: This is a sister directory to the repository -- that is, the code repository and the input/output files both have the same parent directory, which is the "home" variable.
* Avian Flu Files/NCBI Virus: This is the NCBI Virus directory. There are three subdirectories here: "downloads", "temp", and "complete".
* NCBI Virus/downloads: input data
* NCBI Virus/temp: intermediate data created between input and output. A subdirectory is created for this specific date range.
* NCBI Virus/complete: output data

<img src="../Avian_Flu_Files/Presentations/ncbi_virus_file_tree.png" width="500" height="250" alt="NCBI Virus file tree structure">

In [74]:
# Paths

home = Path.home() / "OneDrive - National Institutes of Health/Documents/Virus_Evolution/"
avian_flu_files = home / "Avian_Flu_Files" # Avian flu files is a sister directory of the directory this code is housed in
references = home / "Avian_Flu" / "references"
downloads = Path.home() / "Downloads"

downloads_saved = avian_flu_files / "NCBI_Virus/downloads" / (date_range + "_" + locations.replace(", ", "_").replace(" ", "_"))  

temp_files = avian_flu_files / "NCBI_Virus/temp" / ("segments_" + date_range)
if not temp_files.exists():
    Path.mkdir(temp_files, parents=True, exist_ok=True)
complete_files = avian_flu_files / "NCBI_Virus/complete" / (date_range + "_" + locations.replace(", ", "_").replace(" ", "_"))
if not complete_files.exists(): # checking if the directory exists or not
    Path.mkdir(complete_files, parents=True, exist_ok=True) # if the directory is not present then create it

states_ref = pd.read_csv(references / "states_ref.csv")

genotypes_df = pd.read_excel(references / "genotype_key.xlsx")
genotypes = list(genotypes_df["Genotype"].values)
genotypes.append("Not") 

print(genotypes)

['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'B1.1', 'B1.2', 'B1.3', 'B2.1', 'B2.2', 'B3.1', 'B3.2', 'B3.3', 'B3.4', 'B3.5', 'B3.6', 'B4.1', 'B5.1', 'Minor01', 'Minor04', 'Minor07', 'Minor08', 'Minor09', 'Minor10', 'Minor11', 'Minor12', 'Minor13', 'Minor14', 'Minor15', 'Minor16', 'Minor17', 'Minor18', 'Minor19', 'Minor24', 'Minor25', 'Minor26', 'Minor27', 'Minor28', 'Minor29', 'Minor30', 'Minor31', 'Minor32', 'Minor33', 'Minor34', 'Minor35', 'Minor36', 'Minor37', 'Minor38', 'Minor39', 'Minor40', 'Minor41', 'Minor42', 'Minor43', 'Minor44', 'Minor45', 'Minor46', 'Minor47', 'Minor48', 'B3.7', 'Minor50', 'Minor51', 'C1.1', 'Minor52', 'Minor53', 'B3.11', 'Minor55', 'Minor56', 'Minor57', 'Minor58', 'B3.10', 'C2.1', 'Minor60', 'Minor61', 'B3.8', 'Minor62', 'Minor63', 'B3.12', 'Minor65', 'Minor66', 'Minor67', 'B3.9', 'Minor70', 'Minor71', 'B3.13', 'Minor73', 'Minor74', 'Minor75', 'Minor76', 'Minor77', 'Minor78', 'Minor79', 'Minor80', 'Minor81', 'Minor82', 'Minor83', 'Minor84', 'C3.1', 'Minor86', 'Mino

## Downloading Data

In [4]:
os.chdir(downloads)

if not downloads_saved.exists(): # checking if the directory exists or not
    Path.mkdir(downloads_saved, parents=True, exist_ok=True) # if the directory is not present then create it

# Move downloaded files to saved downloads
for dirpath, dirs, files in os.walk(downloads_saved):
    if len(files) != 0: # If we have downloaded files already, skip
        break 
    else: # If we don't have any downloaded files, get files
        for dirpath, dirs, files in os.walk(downloads):
            if len(files) > 0: # If we have any files that need to be moved
                for file in files:
                    file_name = os.path.join(dirpath, file)
                    destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                    try:
                        shutil.move(file_name, destination_path)
                    except:
                        print("Error moving file", file_name)
                        continue 
            break 
    break 

## De-Duplication

In [5]:
# Get metadata
os.chdir(downloads_saved)
metadata = pd.read_csv("sequences.csv")
print(len(metadata))

# Make sure we only have completed sequences -- 8 segments each 

metadata_counts = metadata.groupby(metadata.Assembly, as_index=False).size()
print(metadata_counts)
metadata_counted = metadata.merge(metadata_counts, on="Assembly")

# Only keep those with size >= 8

metadata_complete_segs = metadata_counted[metadata_counted["size"] >= 8] # May have duplicates
metadata_complete_segs = metadata_counted.drop_duplicates(subset="GenBank_Title", keep="last") # Get rid of duplicate segments

# Now only accept == 8 segments

metadata_segments = metadata_complete_segs[metadata_complete_segs["size"] == 8]
metadata_segments = metadata_complete_segs
metadata_segments

167143
              Assembly  size
0      GCA_038929245.1     8
1      GCA_038932225.1     8
2      GCA_038932275.1     8
3      GCA_038932295.1     8
4      GCA_038933705.1     8
...                ...   ...
19181  GCA_059952155.1     8
19182  GCA_059952165.1     8
19183  GCA_059952175.1     8
19184  GCA_059952205.1     8
19185  GCA_059952255.1     8

[19186 rows x 2 columns]


,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Host,Tissue_Specimen_Source,Submitters,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size
0,PZ803360.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Branta hutchinsii,"oronasopharynx, feces",Direct Submission,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8
1,PZ803361.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Branta hutchinsii,"oronasopharynx, feces",Direct Submission,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8
2,PZ803362.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Branta hutchinsii,"oronasopharynx, feces",Direct Submission,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8
3,PZ803363.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Branta hutchinsii,"oronasopharynx, feces",Direct Submission,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8
4,PZ803364.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Branta hutchinsii,"oronasopharynx, feces",Direct Submission,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153467,OK205699.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8
153468,OK205700.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8
153469,OK205701.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8
153470,OK205702.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8


## Add sequences to dataframe

In [6]:
os.chdir(downloads_saved)

sequences_fasta = df_from_fasta("sequences.fasta") # Turn fasta into a dataframe

sequences_fasta["Accession"] = sequences_fasta["full_header"].apply(lambda x: x.split(" |")[0].replace(">",""))

print(sequences_fasta["full_header"])

# Double-check the de-duplication
print(len(sequences_fasta)) 
print(len(metadata_segments))

# Add sequences to the dataframe
metadata_segments = pd.merge(metadata_segments, sequences_fasta, on="Accession") # , "Segment"])

0         >PZ803346.1 |Influenza A virus (A/Canada Goose...
1         >PZ803347.1 |Influenza A virus (A/Canada Goose...
2         >PZ803348.1 |Influenza A virus (A/Canada Goose...
3         >PZ803349.1 |Influenza A virus (A/Canada Goose...
4         >PZ803350.1 |Influenza A virus (A/Canada Goose...
                                ...                        
167138    >OK205883.1 |Influenza A virus (A/chicken/Vera...
167139    >OK205884.1 |Influenza A virus (A/chicken/Vera...
167140    >OK205885.1 |Influenza A virus (A/chicken/Vera...
167141    >OK205886.1 |Influenza A virus (A/chicken/Vera...
167142    >OK205887.1 |Influenza A virus (A/chicken/Vera...
Name: full_header, Length: 167143, dtype: object
167143
151960


In [7]:
metadata_segments

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Submitters,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size,full_header,sequence
0,PZ803360.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Direct Submission,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8,>PZ803360.1 |Influenza A virus (A/Cackling Goo...,ATGGAAAACATAGTACTTCTTCTTGCAATAATTAGCCTTGTTAAAA...
1,PZ803361.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Direct Submission,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8,>PZ803361.1 |Influenza A virus (A/Cackling Goo...,AAAGCAGGTAGATATTGAAAGATGAGTCTTCTAACCGAGGTCGAAA...
2,PZ803362.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Direct Submission,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8,>PZ803362.1 |Influenza A virus (A/Cackling Goo...,AAAGCAGGGTGACAAAAACATAATGGATTCCAACACTGTGTCAAGC...
3,PZ803363.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Direct Submission,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8,>PZ803363.1 |Influenza A virus (A/Cackling Goo...,AAAGCAGGCAAACCATTTGAATGGATGTCAATCCGACTTTACTTTT...
4,PZ803364.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Direct Submission,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8,>PZ803364.1 |Influenza A virus (A/Cackling Goo...,ATGAATCCAAATCAAAAGATAACAACTATCGGGTCAATCTGCATGG...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151955,OK205699.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205699.1 |Influenza A virus (A/chicken/Mexi...,CAACTGTCAAAATGGAAAGGATAGTAATTGCCTTTGCAATAATCAG...
151956,OK205700.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205700.1 |Influenza A virus (A/chicken/Mexi...,GTAGATAATCACTCACTGAGTGACATTCACATCATGGCGTCTCAAG...
151957,OK205701.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205701.1 |Influenza A virus (A/chicken/Mexi...,ATGAATCCAAATCAGAAAATACTAACAATAGGCTCCACCTCTCTAA...
151958,OK205702.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205702.1 |Influenza A virus (A/chicken/Mexi...,TAGATGTTGAAAGATGAGTCTTCTAACCGAGGTCGAAACGTACGTT...


## Find genotypes

Set up files that are friendly to multi-genoflu, then run multi-genoflu.

In [8]:
metadata_segments["Partial_Header_temp"] = metadata_segments["Isolate"] # Get only isolate

for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]: # Forbidden punctuation
    metadata_segments["Partial_Header_temp"] = metadata_segments["Partial_Header_temp"].apply(lambda x: x.replace(c, "_") if x == x else x)

print(len(metadata_segments))

151960


### Create FASTA files of unknown genotypes 

In [9]:
metadata_segments

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size,full_header,sequence,Partial_Header_temp
0,PZ803360.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8,>PZ803360.1 |Influenza A virus (A/Cackling Goo...,ATGGAAAACATAGTACTTCTTCTTGCAATAATTAGCCTTGTTAAAA...,A_Cackling_Goose_North_Dakota_2740_2026
1,PZ803361.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8,>PZ803361.1 |Influenza A virus (A/Cackling Goo...,AAAGCAGGTAGATATTGAAAGATGAGTCTTCTAACCGAGGTCGAAA...,A_Cackling_Goose_North_Dakota_2740_2026
2,PZ803362.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8,>PZ803362.1 |Influenza A virus (A/Cackling Goo...,AAAGCAGGGTGACAAAAACATAATGGATTCCAACACTGTGTCAAGC...,A_Cackling_Goose_North_Dakota_2740_2026
3,PZ803363.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8,>PZ803363.1 |Influenza A virus (A/Cackling Goo...,AAAGCAGGCAAACCATTTGAATGGATGTCAATCCGACTTTACTTTT...,A_Cackling_Goose_North_Dakota_2740_2026
4,PZ803364.1,GenBank,GCA_059951935.1,NaN,NaN,PRJNA297868,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,St. Jude Center of Excellence for Influenza Re...,USA,NaN,2026-03-24,2026-08-07,ssRNA(-),8,>PZ803364.1 |Influenza A virus (A/Cackling Goo...,ATGAATCCAAATCAAAAGATAACAACTATCGGGTCAATCTGCATGG...,A_Cackling_Goose_North_Dakota_2740_2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151955,OK205699.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205699.1 |Influenza A virus (A/chicken/Mexi...,CAACTGTCAAAATGGAAAGGATAGTAATTGCCTTTGCAATAATCAG...,1864
151956,OK205700.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205700.1 |Influenza A virus (A/chicken/Mexi...,GTAGATAATCACTCACTGAGTGACATTCACATCATGGCGTCTCAAG...,1864
151957,OK205701.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205701.1 |Influenza A virus (A/chicken/Mexi...,ATGAATCCAAATCAGAAAATACTAACAATAGGCTCCACCTCTCTAA...,1864
151958,OK205702.1,GenBank,GCA_039174385.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,2019-09-03,2021-11-01,ssRNA(-),8,>OK205702.1 |Influenza A virus (A/chicken/Mexi...,TAGATGTTGAAAGATGAGTCTTCTAACCGAGGTCGAAACGTACGTT...,1864


In [10]:
# Create 8 fasta files per segment

# Get all the segments
metadata_segments["Partial_Header"] = metadata_segments["Partial_Header_temp"].apply(lambda x: ">" + x if x == x else x) # .apply(lambda x: x.split("|")[-1].split(")")[0]) # Get only genbank name
metadata_segments = metadata_segments.dropna(subset="Partial_Header")

print(metadata_segments["Partial_Header"].values[0:5])

['>A_Cackling_Goose_North_Dakota_2740_2026'
 '>A_Cackling_Goose_North_Dakota_2740_2026'
 '>A_Cackling_Goose_North_Dakota_2740_2026'
 '>A_Cackling_Goose_North_Dakota_2740_2026'
 '>A_Cackling_Goose_North_Dakota_2740_2026']


In [11]:
# Create list of dataframes
df_list = []
for partial_header in list(set(metadata_segments["Partial_Header"].values)): # Unique partial headers only
    # Get smaller dataframe
    df = metadata_segments[metadata_segments["Partial_Header"] == partial_header]
    df = df.sort_values(by="Segment")
    # Make sure there are 8 segments
    if len(df) == 8:
        # Forbidden characters in headers
        for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]:
            df["Partial_Header"] = df["Partial_Header"].apply(lambda x: x.replace(c, "_"))
            
        df["full_header"] = df["Partial_Header"]
        df_list.append(df)

print(metadata_segments["Partial_Header"])

print(df_list[0]["full_header"].values[:10])

# Make fasta files
for df in df_list:
    segments = list(set(df["Segment"].apply(lambda x: int(x)).values))
    for segment in segments:
        one_row = df[df["Segment"] == segment]
        df_to_fasta(one_row, str(segment) + "_seg.fasta", temp_files)

KeyboardInterrupt: 

### Re-Labeling Using GenoFlu

**STOP HERE AND USE GENOFLU TO FIND NEW GENOTYPES.** Then, make sure "results.tsv" is in the NCBI_Virus downloads directory. <br>

To avoid job kill:
```
sinteractive
```
To activate genoflu conda environment in BioWulf:
```
source myconda
conda activate genoflu
```

To run GenoFLU-multi, change directories to Multi-GenoFLU directory:
``` 
cd GenoFLU-multi
```

And then call the python script:

``` 
python bin/genoflu-multi.py -f <FASTA_directory>
```

In [ ]:
# Ensure that user does the above
input("Use genoflu. Afterwards, press ENTER to continue.")

''

In [84]:
# Merging

os.chdir(downloads_saved)

# Read in genoflu results
output_genoflu = pd.read_csv("results.tsv", delimiter="\t") # This also shows the IDs of the new sequences -- incorporate into sequences report

# Create partial headers to merge genoflu results with previously unknown segments
metadata_segments["Partial_Header_Merge"] = metadata_segments["Partial_Header"].apply(lambda x: x.replace(">", ""))
for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]: # Forbidden punctuation
    metadata_segments["Partial_Header_Merge"] = metadata_segments["Partial_Header_Merge"].apply(lambda x: x.replace(c, "_") if x==x else x)

# Reformat so we can merge
output_genoflu["Partial_Header_Merge"] = output_genoflu["Strain"] #.apply(lambda x: re.split(r'Influenza_A_virus__\D*__\D*_', x)[-1])

# Merge
metadata_genoflu = metadata_segments.merge(output_genoflu, how="inner", on="Partial_Header_Merge") #, suffixes=('_left', '_right')) 
metadata_genoflu = metadata_genoflu.rename(columns={"Genotype_y":"Genotype", "Genotype_x":"Serotype"})

print(metadata_genoflu)


C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_3256\3183371152.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_segments["Partial_Header_Merge"] = metadata_segments["Partial_Header"].apply(lambda x: x.replace(">", ""))
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_3256\3183371152.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_segments["Partial_Header_Merge"] = metadata_segments["Partial_Header_Merge"].apply(lambda x: x.replace(c, "_") if x==x else x)


         Accession GenBank_RefSeq         Assembly SRA_Accession BioSample  \
0       PZ803360.1        GenBank  GCA_059951935.1           NaN       NaN   
1       PZ803361.1        GenBank  GCA_059951935.1           NaN       NaN   
2       PZ803362.1        GenBank  GCA_059951935.1           NaN       NaN   
3       PZ803363.1        GenBank  GCA_059951935.1           NaN       NaN   
4       PZ803364.1        GenBank  GCA_059951935.1           NaN       NaN   
...            ...            ...              ...           ...       ...   
151883  OK205699.1        GenBank  GCA_039174385.1           NaN       NaN   
151884  OK205700.1        GenBank  GCA_039174385.1           NaN       NaN   
151885  OK205701.1        GenBank  GCA_039174385.1           NaN       NaN   
151886  OK205702.1        GenBank  GCA_039174385.1           NaN       NaN   
151887  OK205703.1        GenBank  GCA_039174385.1           NaN       NaN   

         BioProject      Organism_Name                         

In [85]:
# Cut down to only columns we want
metadata_genoflu = metadata_genoflu[["Accession", "Assembly", "GenBank_Title", "Host", "Collection_Date", "SRA_Accession", "Isolate", "Genotype", "Geo_Location", "full_header", "sequence", "Serotype", "Segment", "Partial_Header_Merge"]] #, "Strain"]]

# Get genbank strain name
 
metadata_genoflu["genbank_name"] = metadata_genoflu["GenBank_Title"].apply(lambda x: x.split("(")[1] if x == x else x)
metadata_genoflu["Host"] = metadata_genoflu["genbank_name"].apply(lambda x: x.split("/")[1] if x == x else x)

metadata_genoflu = metadata_genoflu.dropna(subset="genbank_name") 

In [86]:
metadata_genoflu

# Make sure we only have the serotype(s) we want -- this code only works for a single serotype, so change it if multiple are ever needed
for serotype in serotypes:
    metadata_genoflu = metadata_genoflu[metadata_genoflu["Serotype"] == serotype]

## Relabeling Sequences

We want:
* Host
* Geo-Location
* Isolate
* Year
* Collection Date
* Host Type
* Genotype

After running the below code, **STOP TO CHECK** if any new animals appear

In [87]:
# Animals 

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genoflu)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print("New animals to add to reference:", different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, write code dealing with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv", index=False) # Make sure name is different to avoid overwriting the first reference 


['snowy egret', 'double-crested cormorant', 'wyoming', 'peacock', 'irere', 'humboldt penguin', 'silver pheasant', 'megascops choliba', 'duck', 'guanay cormorant', 'seabird', 'whooper swan', 'chinese goose', 'redhead', 'short-eared owl', 'bottlenose dolphin', 'common tern', 'striped skunk', 'great-tailed grackle', 'common eider', 'greater scaup', 'blackbird', 'thalasseus maximus', 'serval', 'royal tern', 'american robin', 'gyrfalcon', 'american crow', 'wood duck', 'great grebe', 'norway rat', 'domestic turkey', 'arctic tern', 'sharp-shinned hawk', 'poultry', 'washington', 'bird', 'bear', 'great blue heron', 'snow goose', 'antofagasta', 'american widgeon', 'green winged teal', 'michigan', 'lesser scaup', 'dunlin', 'crested caracara', 'ring-necked duck', 'mottled duck', 'willet', 'turkey vulture', 'layer chicken', 'barn owl', 'penguin', 'ermine', 'ohio', 'common raven', 'northern elephant seal', 'tiger', 'neotropic cormorant', 'aves', 'rock pigeon', 'golden eagle', 'mergus', 'great horned

In [ ]:
input("Check animals output. Afterwards, press ENTER to continue.")

''

In [88]:
# Re-label sequences with no assigned genotype as "Unassigned"

metadata_genoflu["Genotype"] = metadata_genoflu["Genotype"].apply(lambda x: "Not" if "Not assigned" in str(x) else x) # Unassigned segments are labeled "Unassigned"

# Cut down dataframe only to genotypes we want

metadata_genoflu = metadata_genoflu[metadata_genoflu['Genotype'].isin(genotypes) | (metadata_genoflu["Genotype"].str.contains("Not"))]


In [89]:
metadata_genoflu

,Accession,Assembly,GenBank_Title,Host,Collection_Date,SRA_Accession,Isolate,Genotype,Geo_Location,full_header,sequence,Serotype,Segment,Partial_Header_Merge,genbank_name
0,PZ803360.1,GCA_059951935.1,Influenza A virus (A/Cackling Goose/North Dako...,Cackling Goose,2026-03-24,NaN,A/Cackling Goose/North Dakota/2740/2026,Not,USA,>PZ803360.1 |Influenza A virus (A/Cackling Goo...,ATGGAAAACATAGTACTTCTTCTTGCAATAATTAGCCTTGTTAAAA...,H5N1,4,A_Cackling_Goose_North_Dakota_2740_2026,A/Cackling Goose/North Dakota/2740/2026
1,PZ803361.1,GCA_059951935.1,Influenza A virus (A/Cackling Goose/North Dako...,Cackling Goose,2026-03-24,NaN,A/Cackling Goose/North Dakota/2740/2026,Not,USA,>PZ803361.1 |Influenza A virus (A/Cackling Goo...,AAAGCAGGTAGATATTGAAAGATGAGTCTTCTAACCGAGGTCGAAA...,H5N1,7,A_Cackling_Goose_North_Dakota_2740_2026,A/Cackling Goose/North Dakota/2740/2026
2,PZ803362.1,GCA_059951935.1,Influenza A virus (A/Cackling Goose/North Dako...,Cackling Goose,2026-03-24,NaN,A/Cackling Goose/North Dakota/2740/2026,Not,USA,>PZ803362.1 |Influenza A virus (A/Cackling Goo...,AAAGCAGGGTGACAAAAACATAATGGATTCCAACACTGTGTCAAGC...,H5N1,8,A_Cackling_Goose_North_Dakota_2740_2026,A/Cackling Goose/North Dakota/2740/2026
3,PZ803363.1,GCA_059951935.1,Influenza A virus (A/Cackling Goose/North Dako...,Cackling Goose,2026-03-24,NaN,A/Cackling Goose/North Dakota/2740/2026,Not,USA,>PZ803363.1 |Influenza A virus (A/Cackling Goo...,AAAGCAGGCAAACCATTTGAATGGATGTCAATCCGACTTTACTTTT...,H5N1,2,A_Cackling_Goose_North_Dakota_2740_2026,A/Cackling Goose/North Dakota/2740/2026
4,PZ803364.1,GCA_059951935.1,Influenza A virus (A/Cackling Goose/North Dako...,Cackling Goose,2026-03-24,NaN,A/Cackling Goose/North Dakota/2740/2026,Not,USA,>PZ803364.1 |Influenza A virus (A/Cackling Goo...,ATGAATCCAAATCAAAAGATAACAACTATCGGGTCAATCTGCATGG...,H5N1,6,A_Cackling_Goose_North_Dakota_2740_2026,A/Cackling Goose/North Dakota/2740/2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150779,OP221403.1,GCA_039270135.1,Influenza A virus (A/bald eagle/Florida/W22-18...,bald eagle,2022-03-03,NaN,A/bald eagle/Florida/W22-189/2022,Not,USA,>OP221403.1 |Influenza A virus (A/bald eagle/F...,AGCAAAAGCAGGTACTGATCCGAAATGGAAGACTTTGTGCGACAAT...,H5N1,3,A_bald_eagle_Florida_W22_189_2022,A/bald eagle/Florida/W22-189/2022
150780,OP221404.1,GCA_039270135.1,Influenza A virus (A/bald eagle/Florida/W22-18...,bald eagle,2022-03-03,NaN,A/bald eagle/Florida/W22-189/2022,Not,USA,>OP221404.1 |Influenza A virus (A/bald eagle/F...,AGCAAAAGCAGGTCAATTATATTCAATATGGAGAGAATAAAAGAGC...,H5N1,1,A_bald_eagle_Florida_W22_189_2022,A/bald eagle/Florida/W22-189/2022
150781,OP222196.1,GCA_039271695.1,Influenza A virus (A/bald eagle/Florida/W22-19...,bald eagle,2022-03-08,NaN,A/bald eagle/Florida/W22-191/2022,Not,USA,>OP222196.1 |Influenza A virus (A/bald eagle/F...,AGCGAAAGCAGGTACTGATCCGAAATGGAAGACTTTGTGCGACAAT...,H5N1,3,A_bald_eagle_Florida_W22_191_2022,A/bald eagle/Florida/W22-191/2022
150782,OP222197.1,GCA_039270135.1,Influenza A virus (A/bald eagle/Florida/W22-18...,bald eagle,2022-03-03,NaN,A/bald eagle/Florida/W22-189/2022,Not,USA,>OP222197.1 |Influenza A virus (A/bald eagle/F...,AGCAAAAGCAGGCAAACCATTTGAATGGATGTCAATCCGACTTTAC...,H5N1,2,A_bald_eagle_Florida_W22_189_2022,A/bald eagle/Florida/W22-189/2022


In [90]:
def geo_location_normalize(geolocation: str) -> str:

    geolocation = geolocation.replace(":", ",") # We will need to split on commas later; e.g. USA: MD -> USA, MD

    country = geolocation.split(",")[0] # Get the first part of the geolocation, aka the country 

    state = geolocation.split(",")[-1] # Get the last part of the geolocation, aka the state

    state = state.strip().replace(" ", "_") # Remove leading and trailing whitespace, make sure that all internal spaces are underscored instead

    country = country.strip().replace(" ", "_") # Remove leading and trailing whitespace, make sure that all internal spaces are underscored instead

    country_state = country + "-" + state # Log needed

    return country_state 

# Format: USA-[state abbreviation], e.g. USA-MD
def geo_location_get(metadata: pd.DataFrame, state_ref_file: str = "states_ref.csv") -> pd.DataFrame:

    state_ref = pd.read_csv(state_ref_file)

    # Normalize the locations

    # metadata["name_state_genbank"] = metadata["genbank_name"].apply(lambda x: x.split("/")[2] if x == x else x)

    metadata["Geo_Location_Normalized"] = metadata["Geo_Location"].apply(geo_location_normalize)

    metadata["Geo_Location_Country"] = metadata["Geo_Location_Normalized"].apply(lambda x: x.split("-")[0])

    # Get country
    metadata["Geo_Location_Country"] = metadata["Geo_Location_Country"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Country'].iloc[0]
                                                                              if len(state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Country']) > 0
                                                                              else state_ref.loc[state_ref["State"].str.contains(x), 'Country'].iloc[0]
                                                                              if len(state_ref.loc[state_ref["State"].str.contains(x), 'Country']) > 0
                                                                              else x)

    # Get state
    metadata["Geo_Location_State_med"] = metadata["Geo_Location_Normalized"].apply(lambda x: x.split("-")[-1])

      
    metadata["Geo_Location_State_USA"] = metadata["Geo_Location_State_med"].apply(lambda x: 
    # # If "x" is the abbreviated state (e.g. "MD")
    state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Abbreviation'].iloc[0]
    if len(state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Abbreviation']) > 0 and len(x) > 0
    # # If "x" is the state name (e.g. "Maryland")
    else state_ref.loc[state_ref["State"].str.contains(x), 'Abbreviation'].iloc[0]
    if len(state_ref.loc[state_ref["State"].str.contains(x), 'Abbreviation']) > 0 and len(x) > 0
    # # If "x" has neither the state abbreviation nor the full state name
    else x)
    

    metadata["Geo_Location_New"] = metadata["Geo_Location_Country"] + "-" + metadata["Geo_Location_State_USA"] 

    # If USA-, delete -
    metadata["Geo_Location_New"] = metadata["Geo_Location_New"].apply(lambda x: x.split("-")[0] 
    if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] 
    else x)

    return metadata

def geo_location_strain_name(metadata, state_ref_file):
    # If there is no state in the metadata, try the strain name
    state_ref = pd.read_csv(state_ref_file)

    metadata["strain_name_state_nonhuman"] = metadata["genbank_name"].apply(lambda x: x.split("/")[2])

    metadata["strain_name_state_human"] = metadata["genbank_name"].apply(lambda x: x.split("/")[1])

    # If nonhuman, do strain name [2]
    metadata["Geo_Location_State_Strain_Name"] = np.where((metadata["Geo_Location_State_USA"] == "USA") & (metadata["Host_Type"] != "human"), metadata["strain_name_state_nonhuman"], metadata["Geo_Location_State_USA"])

    # If human, do strain name [1]
    metadata["Geo_Location_State_Strain_Name"] = np.where((metadata["Geo_Location_State_USA"] == "USA") & (metadata["Host_Type"] == "human"), metadata["strain_name_state_human"], metadata["Geo_Location_State_USA"])

    metadata["Geo_Location_State_Strain_Name"] = metadata["Geo_Location_State_Strain_Name"].apply(geo_location_normalize)

    metadata["Geo_Location_State_Strain_Name_State"] = metadata["Geo_Location_State_Strain_Name"].apply(lambda x: x.split("-")[-1])
    
    metadata["Geo_Location_State_Strain_Name"] = metadata["Geo_Location_State_Strain_Name_State"].apply(lambda x: 
    # # If "x" is the abbreviated state (e.g. "MD")
    state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Abbreviation'].iloc[0]
    if len(state_ref.loc[state_ref["Abbreviation"].str.contains(x), 'Abbreviation']) > 0 and len(x) > 0
    # # If "x" is the state name (e.g. "Maryland")
    else state_ref.loc[state_ref["State"].str.contains(x), 'Abbreviation'].iloc[0]
    if len(state_ref.loc[state_ref["State"].str.contains(x), 'Abbreviation']) > 0 and len(x) > 0
    # # If "x" has neither the state abbreviation nor the full state name, and is just USA
    else ""
    if x == "USA" or x == "United_States"
    
    else x)

    metadata["Geo_Location_New"] = metadata["Geo_Location_Country"] + "-" + metadata["Geo_Location_State_Strain_Name"] 

    # If USA-, delete -
    metadata["Geo_Location_New"] = metadata["Geo_Location_New"].apply(lambda x: x.split("-")[0] 
    if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] 
    else x)

    # Log metadata

    
    return metadata

In [91]:
print(states_ref[states_ref['State'].str.contains("New_York")]["Abbreviation"].values[0])

NY


In [92]:
# Re-Labeling

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Make sure NaN doesn't mess up the whole name
metadata_genoflu = metadata_genoflu.fillna("")
metadata_genoflu["Host"] = metadata_genoflu["Host"].apply(str.lower)

# Fix animals
fix_animals_andersen(metadata_genoflu, animals_ref)

# Get the years
metadata_genoflu["Years"] = metadata_genoflu["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y"))



In [93]:
os.chdir(references)
metadata_genoflu = geo_location_get(metadata_genoflu, "states_ref.csv")

In [94]:
metadata_genoflu = geo_location_strain_name(metadata_genoflu, "states_ref.csv")

In [95]:
# os.chdir(downloads_saved)
# metadata_genoflu.to_csv("metadata_checkpoint.csv")

In [96]:
# If there is no SRA Accession, replace identifier with Assembly 
metadata_genoflu["Identifier"] = metadata_genoflu["Assembly"].apply(lambda x: x if metadata_genoflu[metadata_genoflu["Assembly"] == x]["SRA_Accession"].values[0] == "" else metadata_genoflu[metadata_genoflu["Assembly"] == x]["SRA_Accession"].values[0]) # np.where(metadata_genoflu['SRA_Accession'] != "", metadata_genoflu['SRA_Accession'], metadata_genoflu['Assembly'].apply(lambda x: x.split(".")[0]))

print((metadata_genoflu[metadata_genoflu["Identifier"].str.contains("SRR")])) # metadata_genoflu[(metadata_genoflu["Identifier"].str.contains("GCA")) | 

         Accession         Assembly  \
96      PZ796614.1  GCA_059952205.1   
97      PZ796615.1  GCA_059952205.1   
98      PZ796616.1  GCA_059952205.1   
99      PZ796617.1  GCA_059952205.1   
100     PZ796618.1  GCA_059952205.1   
...            ...              ...   
150083  OQ565628.1  GCA_039342815.1   
150084  OQ565629.1  GCA_039342815.1   
150085  OQ565630.1  GCA_039342815.1   
150086  OQ565631.1  GCA_039342815.1   
150087  OQ565632.1  GCA_039342815.1   

                                            GenBank_Title       Host  \
96      Influenza A virus (A/chicken/PA/26G09412-001-o...    chicken   
97      Influenza A virus (A/chicken/PA/26G09412-001-o...    chicken   
98      Influenza A virus (A/chicken/PA/26G09412-001-o...    chicken   
99      Influenza A virus (A/chicken/PA/26G09412-001-o...    chicken   
100     Influenza A virus (A/chicken/PA/26G09412-001-o...    chicken   
...                                                   ...        ...   
150083  Influenza A virus (

In [97]:

# Make new labels
names = ">" + metadata_genoflu["Identifier"].astype(str) + "|" + metadata_genoflu["genbank_name"] + "|" + metadata_genoflu["Serotype"] + "|" + metadata_genoflu["Geo_Location_New"] + "|" + metadata_genoflu["Collection_Date"].astype(str) + "|" + metadata_genoflu["Host_Type"] + "|" + metadata_genoflu["Genotype"]

metadata_genoflu["Name"] = names



In [98]:
os.chdir(complete_files)

# De-duplicate

print(len(metadata_genoflu))
metadata_genoflu = metadata_genoflu.drop_duplicates(subset=["Isolate", "genbank_name", "Segment"], keep="last")
print(len(metadata_genoflu))

# Save metadata
# metadata_genoflu.to_csv("NCBI_Virus_" + date_range + "_metadata.csv") # Make file for metadata


148457
148457


## Rename segments and make complete FASTA files

In [99]:
# Set up segments

segments = {1:"PB2", 2:"PB1", 3:"PA", 4:"HA", 5:"NP", 6:"NA", 7:"MP", 8:"NS"} # Name segments 
metadata_genoflu["Segment_Name"] = metadata_genoflu["Segment"].apply(lambda x: int(x)).map(segments)

# Separate into several dataframes based on genotype + segment
segment_genotype_dfs = []
for segment in segments.values():
    m_g = metadata_genoflu[metadata_genoflu["Segment_Name"] == segment]
    for genotype in list(set(m_g["Genotype"].values)):
        if genotype in genotypes:
            df = m_g[(m_g["Genotype"] == genotype)] 
            # df.drop_duplicates(subset="SRA_Accession", keep="first", inplace=True)
            segment_genotype_dfs.append(df)
            pair = genotype + "_" + segment
            print(pair)
        # if "Not" in genotype:
        #     df = m_g[m_g["Genotype"].str.contains("Not")]
        #     segment_genotype_dfs.append(df)
        #     pair = "Unassigned_" + segment
        #     print(pair)

D1.1_PB2
B3.2_PB2
Minor79_PB2
B4.1_PB2
Minor14_PB2
Minor65_PB2
Minor07_PB2
D1.2_PB2
Minor87_PB2
Minor104_PB2
Minor97_PB2
Minor73_PB2
Minor34_PB2
Minor60_PB2
Minor04_PB2
Minor19_PB2
C2.1_PB2
A2_PB2
C3.1_PB2
B1.3_PB2
B5.1_PB2
B3.7_PB2
Minor99_PB2
Minor91_PB2
Minor12_PB2
Minor51_PB2
B1.2_PB2
Minor94_PB2
B3.4_PB2
Minor09_PB2
A1_PB2
Minor81_PB2
B3.10_PB2
B3.1_PB2
Minor50_PB2
Minor90_PB2
Not_PB2
A6_PB2
Minor11_PB2
B1.1_PB2
B2.2_PB2
A4_PB2
B3.6_PB2
A3_PB2
A5_PB2
B3.13_PB2
Minor08_PB2
B3.12_PB2
Minor45_PB2
Minor01_PB2
D1.3_PB2
B3.3_PB2
Minor105_PB2
Minor77_PB2
Minor13_PB2
B2.1_PB2
B3.5_PB2
Minor100_PB2
D1.1_PB1
B3.2_PB1
Minor79_PB1
B4.1_PB1
Minor14_PB1
Minor65_PB1
Minor07_PB1
D1.2_PB1
Minor87_PB1
Minor104_PB1
Minor97_PB1
Minor73_PB1
Minor34_PB1
Minor60_PB1
Minor04_PB1
Minor19_PB1
C2.1_PB1
A2_PB1
C3.1_PB1
B1.3_PB1
B5.1_PB1
B3.7_PB1
Minor99_PB1
Minor91_PB1
Minor12_PB1
Minor51_PB1
B1.2_PB1
Minor94_PB1
B3.4_PB1
Minor09_PB1
A1_PB1
Minor81_PB1
B3.10_PB1
B3.1_PB1
Minor50_PB1
Minor90_PB1
Not_PB1
A6_PB

In [100]:
# Create FASTA files

os.chdir(complete_files)

for df in segment_genotype_dfs:
    print(df)
    
    df = df.reset_index()
    if len(df["Genotype"].values[0]) > 0 and "Not" not in df["Genotype"].values[0]: # If we have assigned genotypes, including "Unassigned"
        file_name = df["Genotype"].values[0] + "_" + df["Segment_Name"].values[0] + "_" + date_range + ".fasta"
        output_file = open(complete_files / (file_name), "w")

        for index, row in df.iterrows():
            name = df.loc[index, "Name"]
            name = name.replace(" ", "_")
            # names.append(name)
            sequence = df.loc[index, "sequence"]
            # First is header, second is sequence
            output_file.write(name + "\n")
            output_file.write(sequence + "\n")
    elif len(df["Genotype"].values[0]) > 0 and "Not" in df["Genotype"].values[0]:
        file_name = "Unassigned_" + df["Segment_Name"].values[0] + "_" + date_range + ".fasta"
        output_file = open(complete_files / (file_name), "w")

        for index, row in df.iterrows():
            name = df.loc[index, "Name"]
            name = name.replace(" ", "_")
            # names.append(name)
            sequence = df.loc[index, "sequence"]
            # First is header, second is sequence
            output_file.write(name + "\n")
            output_file.write(sequence + "\n")
        
    output_file.close()

         Accession         Assembly  \
168     PZ768697.1  GCA_059776725.1   
176     PZ768705.1  GCA_059776745.1   
184     PZ768713.1  GCA_059776555.1   
192     PZ768721.1  GCA_059776785.1   
208     PZ768737.1  GCA_059776925.1   
...            ...              ...   
124864  PQ664462.1  GCA_046435685.1   
124872  PQ664470.1  GCA_046436335.1   
125563  PQ615362.1  GCA_046438285.1   
125579  PQ585630.1  GCA_046436165.1   
125643  PQ573561.1  GCA_046435525.1   

                                            GenBank_Title             Host  \
168     Influenza A virus (A/chicken/IN/26G08500-001-o...          chicken   
176     Influenza A virus (A/chicken/IN/26G08500-002-o...          chicken   
184     Influenza A virus (A/Bald Eagle/CA/26G08579-00...       bald eagle   
192     Influenza A virus (A/Red-tailed Hawk/MN/26G087...  red-tailed hawk   
208     Influenza A virus (A/Common Tern/NY/26G08915-0...      common tern   
...                                                   ...      